# Misspecified world — four references against ground truth

Bates (Heston + compound-Poisson jumps), a world no member of the Heston family contains, with the true
law available as an oracle. Design and predictions pre-registered in `taskc/DECISIONS.md` section 27,
written before this ran.

| ref | prior | fitted to |
|---|---|---|
| 1 | **Path DDPM**, frozen recipe, **3 seeds** | 10⁵ Bates-P paths |
| 2 | **Heston**, method of moments (the AAPL fitter) | the same paths |
| 3 | **Bates**, one-day mixture MLE | the same paths |
| 4 | **True Bates-P simulator** | — (oracle) |

**Jumps**: λ = 10/yr, μ_J = −3%, σ_J = 4% — 38.5% of total variance, 56.5% of paths carry one, excess
kurtosis 10.3. **A concession that favours the parametric arms**: κ, ξ, ρ are not identifiable from
21-day paths (across κ = 3, 15, 30 the return sd is 5.800/5.795/5.790%), so arms 2 and 3 are handed them
at truth. That isolates the jump misspecification and hands the parametric arms information the DDPM
must learn.

In [ ]:
PINNED_COMMIT    = "20dd1a8f2f7f6ac064f6da37702535d203065f9b"
NOTEBOOK_VERSION = "bates4-2026.09.25b"
EXPECT_TASKC     = "taskc-2026.09.25d"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment, full fp32 (TF32 off), Drive

In [ ]:
import os, sys, json, math, time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
from dataclasses import replace
import taskc
from taskc.config import CFG, frozen
from taskc.data import PathStandardizer, make_loader
from taskc.ptheta import (make_schedule, build_model, train_ptheta_decay_ema,
                          save_checkpoint, load_checkpoint, sample_ptheta)
from taskc.kl_budget import standardise, solve_beta, kl_of
from taskc.bates import (Bates, BATES_P, simulate_bates, bates_cf, bates_vanilla_targets,
                         build_bates, BATES_EXOTICS, fit_bates_mle, truth_exotics)
from taskc.aapl import fit_heston_mom
from config import (P_PARAMS, SimConfig, CALIB_TESTFUNS, VANILLA_C3,
                    HELDOUT_TESTFUNS, HELDOUT_VANILLAS, KAPPA_Q, THETA_Q)
from heston import simulate
from projection import solve
from constraints import build_heldout
import evaluation as ev

_nb = "notebooks/taskc_14_bates_refs_colab.ipynb"
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb).read(), (
    f"{NOTEBOOK_VERSION} not in this notebook at PINNED_COMMIT ({PINNED_COMMIT[:7]}). "
    f"Stale cached notebook, or the pin was not advanced with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_bates4"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/bates4_local")
os.makedirs(DRIVE, exist_ok=True)
print("taskc:", taskc.__version__, "| device:", DEVICE, "| tf32:", torch.backends.cuda.matmul.allow_tf32)

### Stage 0b — world, configuration, resumable result file

In [ ]:
LAM, MUJ, SJ = 10.0, -0.03, 0.04            # section 27.1
BATES_TRUE = replace(BATES_P, lam=LAM, mu_j=MUJ, sig_j=SJ)
BATES_Q    = replace(BATES_TRUE, drift=BATES_TRUE.r, lam=2*LAM, kappa=KAPPA_Q, theta=THETA_Q)
DYN_TRUE   = (BATES_TRUE.kappa, BATES_TRUE.xi, BATES_TRUE.rho)   # 27.2 concession

N_TRAIN, N_DRAW, N_TRUTH = 100_000, 500_000, 4_000_000
SEEDS = [0, 1, 2]
# 27.7: arms have different KL floors, so a fixed TOTAL budget gives each arm a
# different amount of room above its own floor and confounds distance-to-constraints
# with identification. Budgets are excesses over each arm's own floor.
DKL_LEVELS = [0.02, 0.05]
KL_BUDGET = 0.10          # kept only as a secondary, fixed-total column
# 27.8: z_cap = 8 is a pure jump filter in this world -- every one of the 899 rejected
# training blocks (0.899 %) contains a jump, it removes 1.591 % of all jump-bearing
# blocks, and a 3-sigma_J jump sits at 9.34 z. Censoring the feature under test would
# handicap the DDPM on exactly what the experiment measures. Set from the Bates training
# maximum (16.77) plus margin. The generator guard is the max |Y| < 10 assert in
# measure(), which is the right check for a blown-up sampler.
Z_CAP = 20.0
SKIP_DONE = True
H, DT, T21 = 21, 1/252, 21/252

RUN = frozen(artifact_dir=DRIVE, z_cap=Z_CAP)
RES = os.path.join(DRIVE, "bates4.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook_created_by=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__,
    device=DEVICE, n_train=N_TRAIN, n_draw=N_DRAW, seeds=SEEDS, kl_budget=KL_BUDGET,
    world=dict(lam=LAM, mu_j=MUJ, sig_j=SJ, kappa_q=KAPPA_Q, theta_q=THETA_Q, lam_q=2*LAM),
    z_cap=Z_CAP, dkl_levels=DKL_LEVELS,
    stages={}, timings={})
res["notebook_last_run"] = NOTEBOOK_VERSION
def stamp(d): d["_notebook"] = NOTEBOOK_VERSION; return d
def save(): json.dump(res, open(RES,"w"), indent=1, default=float)
save()
print(f"Bates-P: jump share {100*LAM*(MUJ**2+SJ**2)/BATES_TRUE.variance_rate(T21):.1f}% of variance, "
      f"21d vol {100*np.sqrt(BATES_TRUE.variance_rate(T21)):.2f}%")
print("done:", list(res["stages"]))

### Stage 1 — training paths, truth, and the three fits

One 10⁵-path Bates-P sample is the training set for the DDPM seeds **and** the sample the two parametric
arms are fitted to, so the references differ only in what they do with identical data.

In [ ]:
if "fits" not in res["stages"]:
    t0 = time.time()
    ptrain = simulate_bates(BATES_TRUE, SimConfig(n_paths=N_TRAIN, H=H, dt=DT, seed=20260920), world="P")
    Rtrain = np.diff(np.log(ptrain.S), axis=1)
    hes = fit_heston_mom(Rtrain.ravel(), dt=DT)
    hes = replace(hes, kappa=DYN_TRUE[0], xi=DYN_TRUE[1], rho=DYN_TRUE[2], v0=hes.theta, S0=100.0, r=0.05)
    bat = fit_bates_mle(Rtrain, dt=DT, fix_dynamics=DYN_TRUE)
    res["stages"]["fits"] = stamp(dict(
        heston=dict(theta=hes.theta, v0=hes.v0, kappa=hes.kappa, xi=hes.xi, rho=hes.rho, drift=hes.drift),
        bates=dict(theta=bat.theta, v0=bat.v0, kappa=bat.kappa, xi=bat.xi, rho=bat.rho,
                   lam=bat.lam, mu_j=bat.mu_j, sig_j=bat.sig_j, drift=bat.drift),
        truth=dict(theta=BATES_TRUE.theta, lam=LAM, mu_j=MUJ, sig_j=SJ),
        seconds=time.time()-t0))
    res["timings"]["fits"] = time.time()-t0; save()
F = res["stages"]["fits"]
print("Heston fit :", {k: round(v,5) for k,v in F["heston"].items()},
      f"-> vol {100*math.sqrt(F['heston']['theta']):.2f}% (true diffusive 20.00%)")
print("Bates  fit :", {k: round(v,5) for k,v in F["bates"].items()})
HES_FIT = replace(P_PARAMS, theta=F["heston"]["theta"], v0=F["heston"]["v0"],
                  kappa=F["heston"]["kappa"], xi=F["heston"]["xi"], rho=F["heston"]["rho"],
                  drift=F["heston"]["drift"])
BAT_FIT = replace(BATES_TRUE, theta=F["bates"]["theta"], v0=F["bates"]["v0"], lam=F["bates"]["lam"],
                  mu_j=F["bates"]["mu_j"], sig_j=F["bates"]["sig_j"], drift=F["bates"]["drift"])

if "truth" not in res["stages"]:
    t0 = time.time()
    calib = bates_vanilla_targets(BATES_Q, VANILLA_C3, DT).tolist()
    heldo = bates_vanilla_targets(BATES_Q, HELDOUT_VANILLAS, DT).tolist()
    ex = truth_exotics(BATES_Q, N_TRUTH, seed=90210, H=H, dt=DT)
    fwd = float(np.real(bates_cf(np.array([-1j]), BATES_Q, T21))[0])
    res["stages"]["truth"] = stamp(dict(calib=calib, heldout=heldo, exotics=ex, fwd=fwd,
                                        fwd_exact=100*math.exp(0.05*T21), seconds=time.time()-t0))
    res["timings"]["truth"] = time.time()-t0; save()
TR = res["stages"]["truth"]
print(f"\nBates-Q forward {TR['fwd']:.6f} vs {TR['fwd_exact']:.6f}")
for k,v in TR["exotics"].items(): print(f"   {k:28s} {v['price']:9.5f} +- {v['se']:.5f}")

### Stage 2 — three DDPM seeds on those paths

In [ ]:
std = None
for sd in SEEDS:
    key, ck = f"train_s{sd}", os.path.join(DRIVE, f"ptheta_bates_s{sd}.pt")
    if key not in res["stages"]:
        t0 = time.time()
        p, NJ = simulate_bates(BATES_TRUE, SimConfig(n_paths=N_TRAIN, H=H, dt=DT, seed=20260920),
                               world="P", return_jumps=True)
        Y = np.diff(np.log(p.S), axis=1)
        s_ = PathStandardizer.fit(Y, S0=100.0, r=0.05, dt=DT)
        z = s_.to_z(Y).astype(np.float32)
        mxz = np.abs(z).max(axis=1)
        keep = mxz <= RUN.z_cap
        if "zcap_train" not in res["stages"]:            # 27.8 diagnostic, cap vs jumps
            hj = NJ.sum(axis=1) > 0
            zc = {}
            for cap in (8.0, 10.0, 12.0, 16.0, RUN.z_cap):
                rj = mxz > cap
                zc[f"{cap:g}"] = dict(n_rejected=int(rj.sum()), frac=float(rj.mean()),
                                      frac_of_jump_blocks=float((rj & hj).sum()/max(hj.sum(),1)),
                                      frac_of_rejected_with_jump=float((rj & hj).sum()/max(rj.sum(),1)))
            res["stages"]["zcap_train"] = stamp(dict(
                max_abs_z=float(mxz.max()), frac_with_jump=float(hj.mean()), by_cap=zc,
                cap_used=float(RUN.z_cap)))
            save()
            print(f"z-cap on training blocks: max|z| {mxz.max():.3f}, {hj.mean()*100:.2f}% carry a jump")
            for cap, v in zc.items():
                print(f"   cap {cap:>5s}: {v['n_rejected']:6d} rejected ({v['frac']*100:6.3f}%), "
                      f"{v['frac_of_jump_blocks']*100:6.3f}% of jump blocks, "
                      f"{v['frac_of_rejected_with_jump']*100:5.1f}% of rejected carry a jump")
        cfg = replace(RUN, init_seed=sd)
        m = build_model(cfg).to(DEVICE); sch = make_schedule(cfg, device=DEVICE)
        m = train_ptheta_decay_ema(m, make_loader(z[keep], batch_size=cfg.batch_size, seed=sd),
                                   sch, cfg, device=DEVICE); m.eval()
        save_checkpoint(ck, m, s_, cfg, extra=dict(seed=sd, train_seconds=time.time()-t0))
        res["stages"][key] = stamp(dict(seed=sd, n=int(keep.sum()), rejected=int((~keep).sum()),
                                        seconds=time.time()-t0)); save()
        print(f"seed {sd} trained in {(time.time()-t0)/60:.1f} min")
models = {}
for sd in SEEDS:
    m, s_, ckc, _ = load_checkpoint(os.path.join(DRIVE, f"ptheta_bates_s{sd}.pt"), device=DEVICE)
    assert ckc["epochs"] == 450 and ckc["lr_decay"] and ckc["ema"], "not the frozen recipe"
    models[sd] = m; std = s_
sched = make_schedule(RUN, device=DEVICE)
print("seeds ready:", list(models))

### Stage 3 — draw, project, and measure every reference

Each reference gives a 5×10⁵ draw, projected onto the identical constraint set with Bates-Q CF targets.
Span R², sd(f⊥), the predicted width constant and the 0.10-nat interval come from the same weighted
least-squares and dual machinery used in §E.13 and the KL-budget work.

In [ ]:
def measure(name, paths):
    # A prior that has blown up must stop the run, not report 1e56 as a price.
    Y = np.diff(np.log(paths.S), axis=1)
    bad = int((~np.isfinite(Y)).sum())
    assert bad == 0, f"{name}: {bad} non-finite returns"
    assert np.abs(Y).max() < 10.0, (
        f"{name}: max |daily log-return| {np.abs(Y).max():.3g} -- a factor of e^10 in one day is a "
        f"broken generator, not a fat tail; inspect it rather than pricing with it")
    cs = build_bates(paths, CALIB_TESTFUNS, VANILLA_C3, BATES_Q, DT)
    r = solve(cs); w = r.w
    G = np.asarray(cs.G, dtype=np.float64)
    Gs, c_std, _ = standardise(G, cs.c.astype(np.float64))
    ho = build_heldout(paths, HELDOUT_TESTFUNS, HELDOUT_VANILLAS)
    hm = np.array([k == "vanilla" for k in ho.kinds])
    ho_px = np.array([float(w @ ho.G[:, j]) for j in np.where(hm)[0]])
    ho_rmse = float(np.sqrt(np.mean((ho_px - np.array(TR["heldout"]))**2)))
    sw = np.sqrt(w)
    X = np.empty((Gs.shape[0], Gs.shape[1]+1)); X[:,0] = sw; X[:,1:] = Gs*sw[:,None]
    out = dict(ess=float(r.ess_frac), kl_floor=float(r.kl), m=int(cs.m), heldout_van_rmse=ho_rmse, exotics={})
    beta0, w0, zz, lse0, _ = solve_beta(Gs, c_std, np.zeros(Gs.shape[0]), np.zeros(Gs.shape[1]))
    kl0 = kl_of(w0, zz, lse0)
    # The budget is a cap on TOTAL KL, so it is only meaningful above the floor. A prior
    # whose floor already exceeds it admits no measure at that budget, and reporting a
    # width divided by sqrt(budget - floor) would print a number from a negative root.
    budget_ok = kl0 < KL_BUDGET
    out["kl_floor_dual"] = float(kl0); out["budget_ok"] = bool(budget_ok)
    for k, spec in BATES_EXOTICS.items():
        f = np.asarray(ev.exotic_payoff(paths, **spec), dtype=np.float64)
        coef, *_ = np.linalg.lstsq(X, f*sw, rcond=None)
        resid = f - (coef[0] + Gs @ coef[1:])
        vp = float(w @ (resid*resid)) - float(w @ resid)**2
        vt = float(w @ (f*f)) - float(w @ f)**2
        # a payoff can be identically zero under a given prior (e.g. every path breaches
        # the barrier), which leaves no variance to explain
        r2 = (1 - vp/vt) if vt > 0 else float("nan")
        px = float(w @ f)
        se = float(np.sqrt(np.sum(w**2 * (f - px)**2)))
        # 0.10-nat budgeted interval: walk gamma until total KL hits the budget
        def walk(target):
            pair = {}
            for sgn in (+1.0, -1.0):
                g, beta, hit = sgn*0.25/max(f.std(), 1e-12), beta0.copy(), None
                for _ in range(40):
                    beta, ww, z2, lse2, _ = solve_beta(Gs, c_std, g*f, beta)
                    if kl_of(ww, z2, lse2) >= target: hit = float(ww @ f); break
                    g *= 1.6
                pair["upper" if sgn > 0 else "lower"] = hit
            return pair
        # PRIMARY: excess over this arm's own floor, so every arm gets the same room
        iv = {f"{d:.2f}": walk(kl0 + d) for d in DKL_LEVELS}
        # SECONDARY: the fixed total budget, only meaningful when the floor is below it
        iv["total_%.2f" % KL_BUDGET] = walk(KL_BUDGET) if budget_ok else {"lower": None, "upper": None}
        out["exotics"][k] = dict(price=px, se=se, var_perp=vp, r2=r2,
                                 sd_perp=float(np.sqrt(vp)), const_pred=float(2*np.sqrt(2*vp)),
                                 intervals=iv)
    return out

REFS = {}
if "refs" not in res["stages"]:
    t0 = time.time(); out = {}
    def add(name, paths):
        out[name] = stamp(measure(name, paths)); res["stages"]["refs"] = out; save()
        e = out[name]
        print(f"{name:12s} ESS {e['ess']*100:6.2f}%  KL floor {e['kl_floor']:.4f}  "
              f"heldout RMSE {e['heldout_van_rmse']:.5f}  " +
              "  ".join(f"{k.split('_')[0]} {v['price']:.4f}" for k,v in e["exotics"].items()), flush=True)
    add("oracle", simulate_bates(BATES_TRUE, SimConfig(n_paths=N_DRAW,H=H,dt=DT,seed=4242),
                                 world="P").without_variance())
    add("heston_fit", simulate(HES_FIT, SimConfig(n_paths=N_DRAW,H=H,dt=DT,seed=4243), world="P").without_variance())
    add("bates_fit", simulate_bates(BAT_FIT, SimConfig(n_paths=N_DRAW,H=H,dt=DT,seed=4244),
                                    world="P").without_variance())
    zgen = {}
    for sd in SEEDS:
        d = sample_ptheta(models[sd], sched, n=N_DRAW, seed=5000+sd, cfg=RUN, device=DEVICE, verbose=False)
        zgen[f"ddpm_s{sd}"] = dict(n_rejected=int(d.n_rejected), n_drawn=int(d.n_drawn),
                                   reject_frac=float(d.n_rejected/max(d.n_drawn,1)),
                                   max_abs_z=float(np.abs(d.z).max()), cap=float(RUN.z_cap))
        print(f"   seed {sd} draw: rejected {d.n_rejected}/{d.n_drawn} "
              f"({100*d.n_rejected/max(d.n_drawn,1):.3f}%), max|z| {np.abs(d.z).max():.3f} vs cap {RUN.z_cap:g}")
        add(f"ddpm_s{sd}", std.to_paths(d.z.astype(np.float64), world=f"ddpm_s{sd}"))
    res["stages"]["zcap_generated"] = stamp(zgen); save()
    res["timings"]["refs"] = time.time()-t0; save()
REFS = res["stages"]["refs"]
print("references measured:", list(REFS))

### Stage 4 — the tables

In [ ]:
EX = list(BATES_EXOTICS)
LAB = {"oracle":"oracle (true Bates-P)","heston_fit":"Heston fitted","bates_fit":"Bates fitted",
       "ddpm_s0":"DDPM seed 0","ddpm_s1":"DDPM seed 1","ddpm_s2":"DDPM seed 2"}
band = {k: max(REFS[f"ddpm_s{s}"]["exotics"][k]["price"] for s in SEEDS)
           - min(REFS[f"ddpm_s{s}"]["exotics"][k]["price"] for s in SEEDS) for k in EX}
ddpm_mean = {k: float(np.mean([REFS[f"ddpm_s{s}"]["exotics"][k]["price"] for s in SEEDS])) for k in EX}
print("="*108); print("PROJECTION"); print("="*108)
print(f"{'reference':26s} {'ESS %':>8s} {'KL floor':>10s} {'held-out van RMSE':>19s}")
for a in REFS: print(f"{LAB[a]:26s} {REFS[a]['ess']*100:8.2f} {REFS[a]['kl_floor']:10.4f} "
                     f"{REFS[a]['heldout_van_rmse']:19.5f}")
print("\n" + "="*108); print("EXOTICS  (seed band = max-min over the three DDPM seeds)"); print("="*108)
for k in EX:
    o = REFS["oracle"]["exotics"][k]["price"]; q = TR["exotics"][k]["price"]
    print(f"\n{k}   oracle Q* {o:.5f}   Bates-Q truth {q:.5f} +- {TR['exotics'][k]['se']:.5f}   "
          f"seed band {band[k]:.5f}")
    print(f"   {'reference':26s} {'price':>10s} {'vs oracle':>11s} {'bands':>8s} {'vs Bates-Q':>12s} {'bands':>8s}")
    for a in REFS:
        p = REFS[a]["exotics"][k]["price"]
        print(f"   {LAB[a]:26s} {p:10.5f} {p-o:+11.5f} {abs(p-o)/band[k] if band[k]>0 else float('nan'):8.2f} "
              f"{p-q:+12.5f} {abs(p-q)/band[k] if band[k]>0 else float('nan'):8.2f}")
    print(f"   {'DDPM mean of seeds':26s} {ddpm_mean[k]:10.5f} {ddpm_mean[k]-o:+11.5f} "
          f"{abs(ddpm_mean[k]-o)/band[k]:8.2f} {ddpm_mean[k]-q:+12.5f} {abs(ddpm_mean[k]-q)/band[k]:8.2f}")
print("\n" + "="*118)
print("SPAN, AND INTERVALS AT A FIXED EXCESS OVER EACH ARM'S OWN KL FLOOR (27.7)")
print("="*118)
print(f"{'reference':26s} {'exotic':10s} {'R^2':>8s} {'sd(f_p)':>9s} {'C pred':>8s} " +
      "".join(f"{'dKL '+f'{d:.2f}'+' width':>17s}{'C meas':>10s}" for d in DKL_LEVELS))
for a in REFS:
    for k in EX:
        e = REFS[a]["exotics"][k]
        row = (f"{LAB[a]:26s} {k.split('_')[0]:10s} {e['r2']:8.4f} {e['sd_perp']:9.4f} "
               f"{e['const_pred']:8.4f} ")
        for d in DKL_LEVELS:
            pr = e["intervals"][f"{d:.2f}"]; lo, hi = pr["lower"], pr["upper"]
            if lo is None or hi is None: row += f"{'not reached':>17s}{'':>10s}"
            else: row += f"{hi-lo:17.4f}{(hi-lo)/math.sqrt(d):10.4f}"
        print(row)
print("\nC pred = 2 sqrt(2 Var(f_perp)) is the local prediction AT the floor;")
print("C meas = width / sqrt(dKL) with dKL the EXCESS over that arm's own floor, so every")
print("arm is given the same room and the comparison is not confounded by its distance to")
print("the constraints. KL floors differ across arms and are in the projection table above.")
tot = f"total_{KL_BUDGET:.2f}"
print(f"\nsecondary, fixed TOTAL budget {KL_BUDGET}: ", end="")
print(", ".join(f"{LAB[a].split(' ')[0]}="
                + ("n/a (floor above budget)" if not REFS[a].get("budget_ok", True) else "ok")
                for a in REFS))
print("timings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `bates4.json`. Outcomes go to DECISIONS.md section 27.R against the 27.5 predictions,
including any that are falsified.